### Libraries

In [ ]:
import boto3
import json
import time

### Create S3 Bucket 

In [ ]:
s3 = boto3.client("s3", region_name="ap-southeast-2")

s3.create_bucket(
    Bucket="sena-policy-docs",         
    CreateBucketConfiguration={"LocationConstraint": "ap-southeast-2"}
)

# Block all public access (security best practice)
s3.put_public_access_block(
    Bucket="sena-policy-docs",
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,
        "IgnorePublicAcls": True,
        "BlockPublicPolicy": True,
        "RestrictPublicBuckets": True
    }
)

print("S3 bucket created successfully")

### IAM Role for Bedrock Knowledge Base 

In [ ]:
iam = boto3.client("iam")

BUCKET_NAME = "sena-policy-docs"  

# Step 1: Create the role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

role = iam.create_role(
    RoleName="sena-bedrock-kb-role",
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="Role for Bedrock Knowledge Base"
    # PermissionsBoundary line removed - not needed with full IAM access
)
role_arn = role["Role"]["Arn"]
print(f"Role ARN: {role_arn}") 

# Step 2: Attach permissions policy
policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel"],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": ["s3vectors:*"],
            "Resource": "*"
        }
    ]
}

iam.put_role_policy(
    RoleName="sena-bedrock-kb-role",
    PolicyName="BedrockKBPolicy",
    PolicyDocument=json.dumps(policy)
)

print("IAM role and policy created successfully")
print(f"Save this Role ARN: {role_arn}")

### Create the Bedrock Knowledge Base

#### S3 vector

In [ ]:
s3vectors = boto3.client("s3vectors", region_name="ap-southeast-2")

s3vectors.create_vector_bucket(
    vectorBucketName="sena-s3-vectors-bucket"
)

print("S3 Vectors bucket created")

In [ ]:
storageConfiguration={
    "type": "S3_VECTORS",
    "s3VectorsConfiguration": {
        "vectorBucketArn": "arn:aws:s3express:ap-southeast-2:038848608811:bucket/sena-s3-vectors-bucket",
        "indexName": "my-rag-index"
    }
}

#### Vector bucket policy

In [ ]:
s3vectors = boto3.client("s3vectors", region_name="ap-southeast-2")

policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowBedrockKBRole",
            "Effect": "Allow",
            "Principal": {
                "AWS": "arn:aws:iam::038848608811:role/sena-bedrock-kb-role"
            },
            "Action": "s3vectors:*",
            "Resource": [
                "arn:aws:s3vectors:ap-southeast-2:038848608811:bucket/sena-s3-vectors-bucket",
                "arn:aws:s3vectors:ap-southeast-2:038848608811:bucket/sena-s3-vectors-bucket/*"
            ]
        }
    ]
}

s3vectors.put_vector_bucket_policy(
    vectorBucketName="sena-s3-vectors-bucket",
    policy=json.dumps(policy)
)

print("Vector bucket policy applied successfully")

#### Vector index

In [ ]:
s3vectors = boto3.client("s3vectors", region_name="ap-southeast-2")

s3vectors.create_index(
    vectorBucketName="sena-s3-vectors-bucket",
    indexName="my-rag-index",
    dataType="float32",
    dimension=1024,        # Titan Embeddings V2 outputs 1024 dimensions
    distanceMetric="cosine"
)

print("Vector index created")

#### KB 

In [ ]:
bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

ROLE_ARN      = "arn:aws:iam::038848608811:role/sena-bedrock-kb-role"
BUCKET_NAME   = "sena-policy-docs"
ACCOUNT_ID    = "038848608811"
VECTOR_BUCKET = "sena-s3-vectors-bucket"
INDEX_NAME    = "my-rag-index"

VECTOR_BUCKET_ARN = f"arn:aws:s3vectors:ap-southeast-2:{ACCOUNT_ID}:bucket/{VECTOR_BUCKET}"
INDEX_ARN         = f"arn:aws:s3vectors:ap-southeast-2:{ACCOUNT_ID}:bucket/{VECTOR_BUCKET}/index/{INDEX_NAME}"

kb = bedrock_agent.create_knowledge_base(
    name="sena-rag-kb",
    description="RAG knowledge base for document Q&A",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": "arn:aws:bedrock:ap-southeast-2::foundation-model/amazon.titan-embed-text-v2:0"
        }
    },
    storageConfiguration={
        "type": "S3_VECTORS",
        "s3VectorsConfiguration": {
            "vectorBucketArn": VECTOR_BUCKET_ARN,
            "indexArn": INDEX_ARN         
        }
    }
)

kb_id = kb["knowledgeBase"]["knowledgeBaseId"]
print(f"Knowledge Base ID: {kb_id}")

# Wait for KB to become ACTIVE
print("Waiting for KB to become active...")
while True:
    status = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"]
    print(f"Status: {status}")
    if status == "ACTIVE":
        break
    elif status == "FAILED":
        raise Exception("KB creation failed")
    time.sleep(5)


In [ ]:
KB_ID=kb_id
ds = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID,
    name="sena-s3-docs-source",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": "arn:aws:s3:::sena-policy-docs"
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "HIERARCHICAL",
            "hierarchicalChunkingConfiguration": {
                "levelConfigurations": [
                    {"maxTokens": 1500},
                    {"maxTokens": 300}
                ],
                "overlapTokens": 60
            }
        }
    }
)

ds_id = ds["dataSource"]["dataSourceId"]
print(f"New Data Source ID: {ds_id}")
print("Save this new DS_ID!")

In [ ]:
s3vectors.create_index(
    vectorBucketName="sena-s3-vectors-bucket",
    indexName="my-rag-index",
    dataType="float32",
    dimension=1024,
    distanceMetric="cosine",
    metadataConfiguration={
        "nonFilterableMetadataKeys": [
            "AMAZON_BEDROCK_TEXT",
            "AMAZON_BEDROCK_METADATA"
        ]
    }
)

print("Index recreated with non-filterable metadata keys")

#### Rerank Permission

In [ ]:
import boto3
import json

iam = boto3.client("iam")

# Get existing policy
response = iam.get_role_policy(
    RoleName="sena-bedrock-kb-role",
    PolicyName="BedrockKBPolicy"
)
policy = response["PolicyDocument"]

# Add rerank permission
policy["Statement"].append({
    "Effect": "Allow",
    "Action": ["bedrock:Rerank"],
    "Resource": "*"
})

iam.put_role_policy(
    RoleName="sena-bedrock-kb-role",
    PolicyName="BedrockKBPolicy",
    PolicyDocument=json.dumps(policy)
)

print("Rerank permission added")

### S3 seperation

In [ ]:
import boto3

s3 = boto3.client("s3", region_name="ap-southeast-2")
BUCKET = "sena-policy-docs"
YOUR_FOLDER = "sena/misty/"

# Clean PDFs to keep — move to your folder
clean_pdfs = [
    "Code_of_Conduct_Workers_Draft_May_2018.pdf",
    "Complaints-policy-PDF.pdf",
    "Compliance and enforcement _ NDIS Quality and Safeguards Commission.pdf",
    "NDIS Reportable Incidents Guide.pdf",
    "PB Enquiries Feedback and Complaints Policy.pdf",
    "Participant Safeguarding Policy.pdf",
    "Privacy _ NDIS Quality and Safeguards Commission.pdf",
    "Understanding your Obligations to the NDIS Code of Conduct – supporttoyou.pdf",
    "complainthandlingguidelinesforproviders_0.pdf",
    "ndis-practice-standards-and-quality-indicators (1).pdf",
    "ndis-practice-standards-and-quality-indicators.pdf",
]

# Files to delete (junk)
junk_files = [
    "agent.md",
    "bedrock_agent_instructions.md",
    "bedrock_knowledge_base.md",
    "formatted_apis.json",
    "openapi_schemas.yml",
]

# Move clean PDFs to your folder
for filename in clean_pdfs:
    s3.copy_object(
        Bucket=BUCKET,
        CopySource={"Bucket": BUCKET, "Key": filename},
        Key=f"{YOUR_FOLDER}{filename}"
    )
    s3.delete_object(Bucket=BUCKET, Key=filename)
    print(f"Moved: {filename} → {YOUR_FOLDER}{filename}")

# Delete junk
for filename in junk_files:
    s3.delete_object(Bucket=BUCKET, Key=filename)
    print(f"Deleted: {filename}")

print("\nDone! Verifying structure...")
response = s3.list_objects_v2(Bucket=BUCKET)
for obj in response.get("Contents", []):
    print(obj["Key"])

In [ ]:
import boto3

bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

KB_ID = "KFWSFMVU8U"
DS_ID = "TRRYKD2EK2"  

bedrock_agent.update_data_source(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID,
    name="s3-docs-source",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": "arn:aws:s3:::sena-policy-docs",
            "inclusionPrefixes": ["sena/misty/"]
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "HIERARCHICAL",
            "hierarchicalChunkingConfiguration": {
                "levelConfigurations": [
                    {"maxTokens": 1500},
                    {"maxTokens": 300}
                ],
                "overlapTokens": 60
            }
        }
    }
)
print("Data source updated — only ingesting from sena/misty/")

### Create IAM role for Lambda

In [ ]:
import boto3
import json

iam = boto3.client("iam")

# Trust policy — allows Lambda to assume this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

role = iam.create_role(
    RoleName="sena-misty-ingestion-lambda-role",
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="Lambda role for auto ingestion trigger"
)
role_arn = role["Role"]["Arn"]
print(f"Role ARN: {role_arn}")

# Permissions policy
policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:StartIngestionJob",
                "bedrock:GetIngestionJob"
            ],
            "Resource": f"arn:aws:bedrock:ap-southeast-2:038848608811:knowledge-base/KFWSFMVU8U"
        },
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                "arn:aws:s3:::sena-policy-docs",
                "arn:aws:s3:::sena-policy-docs/sena/misty/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "*"
        }
    ]
}

iam.put_role_policy(
    RoleName="sena-misty-ingestion-lambda-role",
    PolicyName="sena-misty-ingestion-policy",
    PolicyDocument=json.dumps(policy)
)

print("IAM role and policy created")
print(f"Save this Role ARN: {role_arn}")

### Create Lambda for Auto Ingestion

In [ ]:
import boto3
import json
import zipfile
import io
import time

lambda_client = boto3.client("lambda", region_name="ap-southeast-2")

ROLE_ARN = "arn:aws:iam::038848608811:role/sena-misty-ingestion-lambda-role"  
KB_ID    = "KFWSFMVU8U"
DS_ID    = "TRRYKD2EK2"     

# Lambda function code
lambda_code = f"""
import boto3
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

KB_ID = "{KB_ID}"
DS_ID = "{DS_ID}"

def handler(event, context):
    logger.info(f"S3 event received: {{event}}")
    
    for record in event.get("Records", []):
        key = record["s3"]["object"]["key"]
        
        # Only process sena/misty/ uploads
        if not key.startswith("sena/misty/"):
            logger.info(f"Skipping {{key}} — not in sena/misty/")
            continue
            
        if not key.endswith(".pdf"):
            logger.info(f"Skipping {{key}} — not a PDF")
            continue
        
        logger.info(f"Triggering ingestion for: {{key}}")
        
        response = bedrock_agent.start_ingestion_job(
            knowledgeBaseId=KB_ID,
            dataSourceId=DS_ID
        )
        
        job_id = response["ingestionJob"]["ingestionJobId"]
        logger.info(f"Ingestion job started: {{job_id}}")
    
    return {{"statusCode": 200, "body": "Ingestion triggered"}}
"""

# Package as zip
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", lambda_code)
zip_buffer.seek(0)

# Wait for IAM role to propagate
print("Waiting 10 seconds for IAM role to propagate...")
time.sleep(10)

# Create Lambda
response = lambda_client.create_function(
    FunctionName="sena-misty-auto-ingestion",
    Runtime="python3.12",
    Role=ROLE_ARN,
    Handler="lambda_function.handler",
    Code={"ZipFile": zip_buffer.read()},
    Description="Auto-triggers KB ingestion on S3 upload to sena/misty/",
    Timeout=60,
    MemorySize=128,
    Environment={
        "Variables": {
            "KB_ID": KB_ID,
            "DS_ID": DS_ID
        }
    }
)

lambda_arn = response["FunctionArn"]
print(f"Lambda created: {lambda_arn}")
print("Save this Lambda ARN!")

In [ ]:
import boto3

s3 = boto3.client("s3", region_name="ap-southeast-2")
lambda_client = boto3.client("lambda", region_name="ap-southeast-2")

BUCKET_NAME = "sena-policy-docs"
LAMBDA_ARN  = "arn:aws:lambda:ap-southeast-2:038848608811:function:sena-misty-auto-ingestion"

# Give S3 permission to invoke Lambda (new statement ID)
lambda_client.add_permission(
    FunctionName="sena-misty-auto-ingestion",
    StatementId="s3-invoke-permission-v2",  # changed
    Action="lambda:InvokeFunction",
    Principal="s3.amazonaws.com",
    SourceArn=f"arn:aws:s3:::{BUCKET_NAME}"
)

# rest of the code stays the same

# Set S3 event notification
s3.put_bucket_notification_configuration(
    Bucket=BUCKET_NAME,
    NotificationConfiguration={
        "LambdaFunctionConfigurations": [
            {
                "LambdaFunctionArn": LAMBDA_ARN,
                "Events": ["s3:ObjectCreated:*"],
                "Filter": {
                    "Key": {
                        "FilterRules": [
                            {"Name": "prefix", "Value": "sena/misty/"},
                            {"Name": "suffix", "Value": ".pdf"}
                        ]
                    }
                }
            }
        ]
    }
)

print("S3 event notification configured")
print("Any PDF uploaded to sena/misty/ will now auto-trigger ingestion")

In [ ]:
import boto3
import time

bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

KB_ID = "KFWSFMVU8U"
DS_ID = "TRRYKD2EK2"

job = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID
)

job_id = job["ingestionJob"]["ingestionJobId"]
print(f"Job ID: {job_id}")

while True:
    response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DS_ID,
        ingestionJobId=job_id
    )["ingestionJob"]
    status = response["status"]
    print(f"Status: {status}")
    if status == "COMPLETE":
        print(f" Done! Indexed: {response.get('statistics', {}).get('numberOfNewDocumentsIndexed', 'N/A')}")
        break
    elif status == "FAILED":
        print(response.get("failureReasons"))
        raise Exception(" Failed")
    time.sleep(10)

In [ ]:
import boto3

s3vectors = boto3.client("s3vectors", region_name="ap-southeast-2")

response = s3vectors.list_vectors(
    vectorBucketName="sena-s3-vectors-bucket",
    indexName="my-rag-index",
    returnData=False,
    returnMetadata=False,
    maxResults=20
)

print(f"Total vectors returned: {len(response.get('vectors', []))}")
print(f"Next token exists: {'nextToken' in response}")

In [ ]:
import boto3

s3vectors = boto3.client("s3vectors", region_name="ap-southeast-2")

s3vectors.delete_index(
    vectorBucketName="sena-s3-vectors-bucket",
    indexName="my-rag-index"
)
print("Index deleted")

In [ ]:
s3vectors.create_index(
    vectorBucketName="sena-s3-vectors-bucket",
    indexName="my-rag-index",
    dataType="float32",
    dimension=1024,
    distanceMetric="cosine",
    metadataConfiguration={
        "nonFilterableMetadataKeys": [
            "AMAZON_BEDROCK_TEXT",
            "AMAZON_BEDROCK_METADATA"
        ]
    }
)
print("Index recreated clean")

In [ ]:
import time

bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

KB_ID = "KFWSFMVU8U"
DS_ID = "TRRYKD2EK2"

job = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID
)

job_id = job["ingestionJob"]["ingestionJobId"]
print(f"Job ID: {job_id}")

while True:
    response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DS_ID,
        ingestionJobId=job_id
    )["ingestionJob"]
    status = response["status"]
    print(f"Status: {status}")
    if status == "COMPLETE":
        print(f" Done! Indexed: {response.get('statistics', {}).get('numberOfNewDocumentsIndexed', 'N/A')}")
        break
    elif status == "FAILED":
        print(response.get("failureReasons"))
        raise Exception(" Failed")
    time.sleep(10)

In [ ]:
import boto3
import time

bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

KB_ID = "KFWSFMVU8U"
DS_ID = "TRRYKD2EK2"

# Force re-sync by setting a description change
bedrock_agent.update_data_source(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID,
    name="s3-docs-source",
    description="force-resync",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": "arn:aws:s3:::sena-policy-docs",
            "inclusionPrefixes": ["sena/misty/"]
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "HIERARCHICAL",
            "hierarchicalChunkingConfiguration": {
                "levelConfigurations": [
                    {"maxTokens": 1500},
                    {"maxTokens": 300}
                ],
                "overlapTokens": 60
            }
        }
    }
)
print("Data source updated — triggering full re-sync")

# Wait a moment then re-ingest
time.sleep(5)

job = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID
)

job_id = job["ingestionJob"]["ingestionJobId"]
print(f"Job ID: {job_id}")

while True:
    response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DS_ID,
        ingestionJobId=job_id
    )["ingestionJob"]
    status = response["status"]
    print(f"Status: {status}")
    if status == "COMPLETE":
        print(f"✅ Done! Indexed: {response.get('statistics', {}).get('numberOfNewDocumentsIndexed', 'N/A')}")
        break
    elif status == "FAILED":
        print(response.get("failureReasons"))
        raise Exception("❌ Failed")
    time.sleep(10)

In [ ]:
import boto3
import time

bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

KB_ID = "KFWSFMVU8U"
DS_ID = "TRRYKD2EK2"

# Delete data source
bedrock_agent.delete_data_source(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID
)
print("Data source deleted")
time.sleep(5)

In [ ]:

# Recreate data source
ds = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID,
    name="s3-docs-source-v2",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": "arn:aws:s3:::sena-policy-docs",
            "inclusionPrefixes": ["sena/misty/"]
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "HIERARCHICAL",
            "hierarchicalChunkingConfiguration": {
                "levelConfigurations": [
                    {"maxTokens": 1500},
                    {"maxTokens": 300}
                ],
                "overlapTokens": 60
            }
        }
    }
)

new_ds_id = ds["dataSource"]["dataSourceId"]
print(f"New DS ID: {new_ds_id} — SAVE THIS")
time.sleep(5)

# Ingest
job = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=new_ds_id
)

job_id = job["ingestionJob"]["ingestionJobId"]
print(f"Job ID: {job_id}")

while True:
    response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=new_ds_id,
        ingestionJobId=job_id
    )["ingestionJob"]
    status = response["status"]
    print(f"Status: {status}")
    if status == "COMPLETE":
        print(f"✅ Done! Indexed: {response.get('statistics', {}).get('numberOfNewDocumentsIndexed', 'N/A')}")
        break
    elif status == "FAILED":
        print(response.get("failureReasons"))
        raise Exception("❌ Failed")
    time.sleep(10)

In [ ]:
import boto3

lambda_client = boto3.client("lambda", region_name="ap-southeast-2")

lambda_client.update_function_configuration(
    FunctionName="sena-misty-auto-ingestion",
    Environment={
        "Variables": {
            "KB_ID": "KFWSFMVU8U",
            "DS_ID": "CWJ8UCZSCY"
        }
    }
)
print("Lambda updated with new DS ID")

In [ ]:
import boto3
import time

dynamodb = boto3.client("dynamodb", region_name="ap-southeast-2")

# Table 1 — Chat Sessions
dynamodb.create_table(
    TableName="sena-chat-sessions",
    KeySchema=[
        {"AttributeName": "user_id",    "KeyType": "HASH"},
        {"AttributeName": "session_id", "KeyType": "RANGE"}
    ],
    AttributeDefinitions=[
        {"AttributeName": "user_id",    "AttributeType": "S"},
        {"AttributeName": "session_id", "AttributeType": "S"}
    ],
    BillingMode="PAY_PER_REQUEST"
)
print("sena-chat-sessions created")

# Table 2 — Chat Turns
dynamodb.create_table(
    TableName="sena-chat-turns",
    KeySchema=[
        {"AttributeName": "session_id", "KeyType": "HASH"},
        {"AttributeName": "turn_id",    "KeyType": "RANGE"}
    ],
    AttributeDefinitions=[
        {"AttributeName": "session_id", "AttributeType": "S"},
        {"AttributeName": "turn_id",    "AttributeType": "S"}
    ],
    BillingMode="PAY_PER_REQUEST"
)
print("sena-chat-turns created")

# Wait for tables to be active
print("Waiting for tables to be active...")
time.sleep(10)

# Enable TTL on both tables
dynamodb.update_time_to_live(
    TableName="sena-chat-sessions",
    TimeToLiveSpecification={"AttributeName": "ttl", "Enabled": True}
)
dynamodb.update_time_to_live(
    TableName="sena-chat-turns",
    TimeToLiveSpecification={"AttributeName": "ttl", "Enabled": True}
)

print("TTL enabled on both tables")
print("DynamoDB setup complete")

In [ ]:
import boto3
import time

control_client = boto3.client("bedrock-agentcore-control", region_name="ap-southeast-2")

response = control_client.create_memory(
    name="senaPolicyProceduresMemory",
    description="Memory for NDIS policy and procedures chatbot — stores conversation summaries and user preferences",
    eventExpiryDuration=30,  # 30 days retention
    memoryStrategies=[
        {
            "summaryMemoryStrategy": {
                "name": "SessionSummariser",
                "namespaceTemplates": ["/summaries/{actorId}/{sessionId}/"]
            }
        },
        {
            "userPreferenceMemoryStrategy": {
                "name": "PreferenceLearner",
                "namespaceTemplates": ["/preferences/{actorId}/"]
            }
        },
        {
            "semanticMemoryStrategy": {
                "name": "FactExtractor",
                "namespaceTemplates": ["/facts/{actorId}/"]
            }
        }
    ]
)

memory_id = response["memory"]["id"]
print(f"Memory ID: {memory_id}")
print("Waiting for memory to become active (2-3 mins)...")

# Poll until active
while True:
    status = control_client.get_memory(memoryId=memory_id)["memory"]["status"]
    print(f"Status: {status}")
    if status == "ACTIVE":
        print(f" AgentCore Memory ready!")
        print(f"Save this Memory ID: {memory_id}")
        break
    elif status == "FAILED":
        raise Exception(" Memory creation failed")
    time.sleep(15)

In [ ]:
import boto3
import time

dynamodb = boto3.client("dynamodb", region_name="ap-southeast-2")

# Delete existing tables
for table in ["sena-chat-sessions", "sena-chat-turns"]:
    try:
        dynamodb.delete_table(TableName=table)
        print(f"Deleting {table}...")
    except Exception as e:
        print(f"Skip {table}: {e}")

print("Waiting for deletion...")
time.sleep(15)

# Recreate with dev prefix
for table_name, pk, sk in [
    ("sena-dev-chat-sessions", "user_id",    "session_id"),
    ("sena-dev-chat-turns",    "session_id", "turn_id")
]:
    dynamodb.create_table(
        TableName=table_name,
        KeySchema=[
            {"AttributeName": pk, "KeyType": "HASH"},
            {"AttributeName": sk, "KeyType": "RANGE"}
        ],
        AttributeDefinitions=[
            {"AttributeName": pk, "AttributeType": "S"},
            {"AttributeName": sk, "AttributeType": "S"}
        ],
        BillingMode="PAY_PER_REQUEST"
    )
    print(f"Created: {table_name}")

time.sleep(10)

# Enable TTL
for table_name in ["sena-dev-chat-sessions", "sena-dev-chat-turns"]:
    dynamodb.update_time_to_live(
        TableName=table_name,
        TimeToLiveSpecification={"AttributeName": "ttl", "Enabled": True}
    )
    print(f"TTL enabled: {table_name}")

print("✅ Dev tables ready")

In [ ]:
import boto3

s3 = boto3.client("s3", region_name="ap-southeast-2")
BUCKET = "sena-policy-docs"

# List current files in sena/misty/
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="sena/misty/")

for obj in response.get("Contents", []):
    old_key = obj["Key"]
    
    # Skip if already in a subfolder
    if old_key.count("/") > 2:
        print(f"Skipping: {old_key}")
        continue
    
    filename = old_key.split("/")[-1]
    new_key  = f"sena/misty/orgs/ndis/{filename}"
    
    s3.copy_object(
        Bucket=BUCKET,
        CopySource={"Bucket": BUCKET, "Key": old_key},
        Key=new_key
    )
    s3.delete_object(Bucket=BUCKET, Key=old_key)
    print(f"Moved: {old_key} → {new_key}")

print("\nDone! Verifying...")
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="sena/misty/")
for obj in response.get("Contents", []):
    print(obj["Key"])

In [ ]:
import boto3
import os

s3 = boto3.client("s3", region_name="ap-southeast-2")
BUCKET = "sena-policy-docs"

# Change these paths to your dummy doc locations
ORG1_FOLDER = r"C:\Users\BAPS\Documents\SENA_RAG\policies\horizons"   # folder with org1 dummy PDFs
ORG2_FOLDER = r"C:\Users\BAPS\Documents\SENA_RAG\policies\sunrise"   # folder with org2 dummy PDFs

for folder, prefix in [
    (ORG1_FOLDER, "sena/misty/orgs/org_horizons/"),
    (ORG2_FOLDER, "sena/misty/orgs/org_sunrise/")
]:
    for filename in os.listdir(folder):
        if filename.endswith((".pdf", ".docx")):
            filepath = os.path.join(folder, filename)
            s3.upload_file(filepath, BUCKET, f"{prefix}{filename}")
            print(f"Uploaded: {prefix}{filename}")

print("\nVerifying structure...")
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="sena/misty/orgs/")
for obj in response.get("Contents", []):
    print(obj["Key"])

In [ ]:
import boto3
import json

s3 = boto3.client("s3", region_name="ap-southeast-2")
BUCKET = "sena-policy-docs"

# Map folder prefix to org_id
FOLDER_ORG_MAP = {
    "sena/misty/orgs/ndis/":  "ndis",
    "sena/misty/orgs/org_horizons/":  "org_horizons",
    "sena/misty/orgs/org_sunrise/":  "org_sunrise",
}

for prefix, org_id in FOLDER_ORG_MAP.items():
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    
    for obj in response.get("Contents", []):
        key = obj["Key"]
        
        # Skip metadata files themselves
        if key.endswith(".metadata.json"):
            continue
        
        metadata = {
            "metadataAttributes": {
                "org_id": org_id
            }
        }
        
        metadata_key = f"{key}.metadata.json"
        s3.put_object(
            Bucket=BUCKET,
            Key=metadata_key,
            Body=json.dumps(metadata),
            ContentType="application/json"
        )
        print(f"Created: {metadata_key}")

print("\n All metadata files created")

In [ ]:
import boto3
import time

bedrock_agent = boto3.client("bedrock-agent", region_name="ap-southeast-2")

KB_ID = "KFWSFMVU8U"
DS_ID = "CWJ8UCZSCY"

# Update data source to cover all orgs folders
bedrock_agent.update_data_source(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID,
    name="s3-docs-source-v2",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": "arn:aws:s3:::sena-policy-docs",
            "inclusionPrefixes": ["sena/misty/orgs/"]
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "HIERARCHICAL",
            "hierarchicalChunkingConfiguration": {
                "levelConfigurations": [
                    {"maxTokens": 1500},
                    {"maxTokens": 300}
                ],
                "overlapTokens": 60
            }
        }
    }
)
print("Data source updated")
time.sleep(5)

# Delete and recreate index for clean ingest
s3vectors = boto3.client("s3vectors", region_name="ap-southeast-2")

s3vectors.delete_index(
    vectorBucketName="sena-s3-vectors-bucket",
    indexName="my-rag-index"
)
print("Index deleted")
time.sleep(5)

s3vectors.create_index(
    vectorBucketName="sena-s3-vectors-bucket",
    indexName="my-rag-index",
    dataType="float32",
    dimension=1024,
    distanceMetric="cosine",
    metadataConfiguration={
        "nonFilterableMetadataKeys": [
            "AMAZON_BEDROCK_TEXT",
            "AMAZON_BEDROCK_METADATA"
        ]
    }
)
print("Index recreated")
time.sleep(5)

# Ingest
job = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=DS_ID
)
job_id = job["ingestionJob"]["ingestionJobId"]
print(f"Job ID: {job_id}")

while True:
    response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DS_ID,
        ingestionJobId=job_id
    )["ingestionJob"]
    status = response["status"]
    print(f"Status: {status}")
    if status == "COMPLETE":
        print(f" Indexed: {response.get('statistics', {}).get('numberOfNewDocumentsIndexed', 'N/A')}")
        break
    elif status == "FAILED":
        print(response.get("failureReasons"))
        raise Exception(" Failed")
    time.sleep(10)

In [ ]:
import boto3
s3 = boto3.client("s3", region_name="ap-southeast-2")
s3.delete_object(Bucket="sena-policy-docs", Key="sena/misty/orgs/ndis/ndis-practice-standards-and-quality-indicators (1).pdf")
s3.delete_object(Bucket="sena-policy-docs", Key="sena/misty/orgs/ndis/ndis-practice-standards-and-quality-indicators (1).pdf.metadata.json")
print("Deleted duplicate")

## Setup guardrails

In [1]:
#setup_guradrails
import boto3
import json

bedrock = boto3.client("bedrock", region_name="ap-southeast-2")

bedrock.update_guardrail(
    guardrailIdentifier="j9x9dysm5m3h",
    name="sena-rag-guardrail",
    description="Guardrail for NDIS policy chatbot",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "medical-advice",
                "definition": "Specific medical diagnoses, treatment recommendations, or clinical advice for participants.",
                "examples": [
                    "What medication should my participant take?",
                    "Is this symptom serious?",
                    "Should I call a doctor?"
                ],
                "type": "DENY"
            },
            {
                "name": "legal-advice",
                "definition": "Specific legal advice or interpretation of laws beyond NDIS policy documents.",
                "examples": [
                    "Can I sue my provider?",
                    "Am I legally liable?",
                    "What are my legal rights in court?"
                ],
                "type": "DENY"
            },
            {
                "name": "financial-advice",
                "definition": "Personal financial advice or NDIS funding decisions beyond policy guidance.",
                "examples": [
                    "How should I invest my NDIS funds?",
                    "What is the best plan for my money?"
                ],
                "type": "DENY"
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "SEXUAL",        "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "VIOLENCE",      "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "HATE",          "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "INSULTS",       "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "MISCONDUCT",    "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "PROMPT_ATTACK", "inputStrength": "HIGH", "outputStrength": "NONE"},
        ]
    },
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "NAME",      "action": "ANONYMIZE"},
            {"type": "EMAIL",     "action": "ANONYMIZE"},
            {"type": "PHONE",     "action": "ANONYMIZE"},
            {"type": "ADDRESS",   "action": "ANONYMIZE"},
            {"type": "AGE",       "action": "ANONYMIZE"},
            {"type": "DRIVER_ID", "action": "ANONYMIZE"},
            {"type": "PASSWORD",  "action": "BLOCK"},
            {"type": "USERNAME",  "action": "BLOCK"},
        ]
    },
    blockedInputMessaging="I can only help with NDIS policies and organisation procedures. Please ask a relevant question.",
    blockedOutputsMessaging="I wasn't able to generate a response for that. Please rephrase or ask something related to NDIS policies.",
)

print("Guardrail updated — off-topic topic removed")

Guardrail updated — off-topic topic removed


In [ ]:
#versioning guardrails
import boto3

bedrock = boto3.client("bedrock", region_name="ap-southeast-2")

GUARDRAIL_ID = "j9x9dysm5m3h"

response = bedrock.create_guardrail_version(
    guardrailIdentifier=GUARDRAIL_ID,
    description="v1 — initial production version with topic denial, PII, grounding"
)

guardrail_version = response["version"]
print(f"Guardrail version created: {guardrail_version}")
print("Save this version number!")

## Auto Ingestion and auto deletion

### setup_dynamodb_registry.py

In [1]:
# setup_dynamodb_registry.py
# Run once to create the DynamoDB table needed for the doc registry.
# Run from your venv: python setup_dynamodb_registry.py

import boto3
import time

dynamodb = boto3.client("dynamodb", region_name="ap-southeast-2")

TABLE_NAME = "sena-doc-registry"

TABLE_DEF = {
    "TableName": TABLE_NAME,
    "KeySchema": [
        {"AttributeName": "doc_id", "KeyType": "HASH"}   # full S3 key
    ],
    "AttributeDefinitions": [
        {"AttributeName": "doc_id", "AttributeType": "S"},
        {"AttributeName": "org_id", "AttributeType": "S"},
        {"AttributeName": "status", "AttributeType": "S"}
    ],
    "GlobalSecondaryIndexes": [
        {
            # Query all docs for a specific org
            "IndexName": "org_id-index",
            "KeySchema": [
                {"AttributeName": "org_id", "KeyType": "HASH"}
            ],
            "Projection": {"ProjectionType": "ALL"}
        },
        {
            # Query all docs with a specific status (e.g. all FAILED)
            "IndexName": "status-index",
            "KeySchema": [
                {"AttributeName": "status", "KeyType": "HASH"}
            ],
            "Projection": {"ProjectionType": "ALL"}
        }
    ],
    "BillingMode": "PAY_PER_REQUEST"
}


def create_table():
    # Check if already exists
    try:
        dynamodb.describe_table(TableName=TABLE_NAME)
        print(f"Already exists — skipping: {TABLE_NAME}")
        return
    except dynamodb.exceptions.ResourceNotFoundException:
        pass

    print(f"Creating table: {TABLE_NAME}...")
    dynamodb.create_table(**TABLE_DEF)

    # Wait for active
    while True:
        status = dynamodb.describe_table(
            TableName=TABLE_NAME
        )["Table"]["TableStatus"]
        print(f"  Status: {status}")
        if status == "ACTIVE":
            print(f"   {TABLE_NAME} ready")
            break
        time.sleep(5)

    print("\n Table ready")
    print(f"\n  {TABLE_NAME}")
    print("    Partition key : doc_id  (full S3 key)")
    print("    GSI           : org_id-index  — query all docs per org")
    print("    GSI           : status-index  — query all FAILED/INGESTING docs")


if __name__ == "__main__":
    create_table()

Creating table: sena-doc-registry...
  Status: CREATING
  Status: CREATING
  Status: CREATING
  Status: CREATING
  Status: CREATING
  Status: ACTIVE
  ✅ sena-doc-registry ready

✅ Table ready

  sena-doc-registry
    Partition key : doc_id  (full S3 key)
    GSI           : org_id-index  — query all docs per org
    GSI           : status-index  — query all FAILED/INGESTING docs


### setup_eventbridge_upload.py

In [3]:
# setup_eventbridge_upload.py
# Creates the EventBridge rule that triggers auto-ingestion Lambda
# on S3 ObjectCreated events for sena/misty/orgs/ prefix.
#
# Prerequisites:
#   1. EventBridge notifications enabled on S3 bucket (one checkbox —
#      S3 console → sena-policy-docs → Properties → EventBridge → Enable)
#   2. auto_ingestion_lambda deployed and its ARN known
#   3. DynamoDB tables created (setup_dynamodb_registry.py already run)
#
# Run once from your venv:
#   python setup_eventbridge_upload.py

import boto3
import json

events  = boto3.client("events",  region_name="ap-southeast-2")
lambda_ = boto3.client("lambda",  region_name="ap-southeast-2")
iam     = boto3.client("iam")

# ── Config ────────────────────────────────────────────────────────────────────
BUCKET_NAME     = "sena-policy-docs"
ORG_PREFIX      = "sena/misty/orgs/"
LAMBDA_NAME     = "sena-misty-auto-ingestion"   # your existing Lambda name
RULE_NAME       = "sena-s3-upload-to-ingestion"
RULE_DESCRIPTION = "Triggers auto-ingestion Lambda when a doc is uploaded to sena/misty/orgs/"


def get_lambda_arn() -> str:
    response = lambda_.get_function(FunctionName=LAMBDA_NAME)
    arn = response["Configuration"]["FunctionArn"]
    print(f"Lambda ARN: {arn}")
    return arn


def create_eventbridge_rule() -> str:
    """
    Creates EventBridge rule that matches S3 ObjectCreated events
    on sena-policy-docs bucket under sena/misty/orgs/ prefix.
    Returns the rule ARN.
    """

    # Event pattern — matches S3 ObjectCreated for our bucket + prefix
    event_pattern = {
        "source": ["aws.s3"],
        "detail-type": ["Object Created"],
        "detail": {
            "bucket": {
                "name": [BUCKET_NAME]
            },
            "object": {
                "key": [{"prefix": ORG_PREFIX}]
            }
        }
    }

    response = events.put_rule(
        Name=RULE_NAME,
        EventPattern=json.dumps(event_pattern),
        State="ENABLED",
        Description=RULE_DESCRIPTION
    )

    rule_arn = response["RuleArn"]
    print(f"EventBridge rule created: {rule_arn}")
    return rule_arn


def add_lambda_target(lambda_arn: str):
    """
    Adds the auto-ingestion Lambda as the target of the EventBridge rule.
    """
    events.put_targets(
        Rule=RULE_NAME,
        Targets=[
            {
                "Id":  "auto-ingestion-lambda",
                "Arn": lambda_arn
            }
        ]
    )
    print(f"Lambda target added to rule: {RULE_NAME}")


def grant_eventbridge_permission(lambda_arn: str, rule_arn: str):
    """
    Grants EventBridge permission to invoke the Lambda.
    Without this, EventBridge can see the Lambda but cannot call it.
    """
    try:
        lambda_.add_permission(
            FunctionName=LAMBDA_NAME,
            StatementId="eventbridge-s3-upload-invoke",
            Action="lambda:InvokeFunction",
            Principal="events.amazonaws.com",
            SourceArn=rule_arn
        )
        print("Lambda permission granted to EventBridge")
    except lambda_.exceptions.ResourceConflictException:
        # Permission already exists — safe to ignore
        print("Lambda permission already exists — skipping")


def verify_setup():
    """
    Reads back the rule and targets to confirm everything is wired correctly.
    """
    print("\n── Verification ─────────────────────────────────────────────")

    rule = events.describe_rule(Name=RULE_NAME)
    print(f"Rule name:    {rule['Name']}")
    print(f"Rule state:   {rule['State']}")
    print(f"Rule pattern: {rule['EventPattern']}")

    targets = events.list_targets_by_rule(Rule=RULE_NAME)
    for t in targets["Targets"]:
        print(f"Target ID:    {t['Id']}")
        print(f"Target ARN:   {t['Arn']}")

    print("─────────────────────────────────────────────────────────────")


def main():
    print("Setting up EventBridge rule for S3 upload → auto-ingestion Lambda\n")

    # Step 1 — get Lambda ARN
    lambda_arn = get_lambda_arn()

    # Step 2 — create EventBridge rule
    rule_arn = create_eventbridge_rule()

    # Step 3 — add Lambda as target
    add_lambda_target(lambda_arn)

    # Step 4 — grant EventBridge permission to invoke Lambda
    grant_eventbridge_permission(lambda_arn, rule_arn)

    # Step 5 — verify
    verify_setup()

    print("\n EventBridge rule setup complete")
    print("\nReminder: Make sure EventBridge notifications are enabled on")
    print(f"the S3 bucket '{BUCKET_NAME}' — S3 console → Properties → EventBridge → Enable")
    print("\nUpload flow is now fully wired:")
    print(f"  S3 upload to {ORG_PREFIX}*")
    print(f"  → EventBridge rule: {RULE_NAME}")
    print(f"  → Lambda: {LAMBDA_NAME}")
    print(f"  → metadata sidecar created")
    print(f"  → ingestion job triggered")
    print(f"  → registry updated")


if __name__ == "__main__":
    main()

Setting up EventBridge rule for S3 upload → auto-ingestion Lambda

Lambda ARN: arn:aws:lambda:ap-southeast-2:038848608811:function:sena-misty-auto-ingestion
EventBridge rule created: arn:aws:events:ap-southeast-2:038848608811:rule/sena-s3-upload-to-ingestion
Lambda target added to rule: sena-s3-upload-to-ingestion
Lambda permission granted to EventBridge

── Verification ─────────────────────────────────────────────
Rule name:    sena-s3-upload-to-ingestion
Rule state:   ENABLED
Rule pattern: {"source": ["aws.s3"], "detail-type": ["Object Created"], "detail": {"bucket": {"name": ["sena-policy-docs"]}, "object": {"key": [{"prefix": "sena/misty/orgs/"}]}}}
Target ID:    auto-ingestion-lambda
Target ARN:   arn:aws:lambda:ap-southeast-2:038848608811:function:sena-misty-auto-ingestion
─────────────────────────────────────────────────────────────

 EventBridge rule setup complete

Reminder: Make sure EventBridge notifications are enabled on
the S3 bucket 'sena-policy-docs' — S3 console → Pro

### setup_eventbridge_delete.py

In [ ]:
# setup_eventbridge_delete.py
# Creates the EventBridge rule that triggers cleanup Lambda
# on S3 ObjectRemoved events for sena/misty/orgs/ prefix.
#
# Prerequisites:
#   1. EventBridge notifications enabled on S3 bucket (same checkbox
#      used for upload rule — already done if you ran setup_eventbridge_upload.py)
#   2. cleanup_lambda deployed and its ARN known
#   3. DynamoDB tables already created (setup_dynamodb_registry.py already run)
#
# Run once from your venv:
#   python setup_eventbridge_delete.py

import boto3
import json

events  = boto3.client("events",  region_name="ap-southeast-2")
lambda_ = boto3.client("lambda",  region_name="ap-southeast-2")

# ── Config ────────────────────────────────────────────────────────────────────
BUCKET_NAME      = "sena-policy-docs"
ORG_PREFIX       = "sena/misty/orgs/"
LAMBDA_NAME      = "sena-misty-cleanup"        # name you deploy cleanup_lambda.py as
RULE_NAME        = "sena-s3-delete-to-cleanup"
RULE_DESCRIPTION = "Triggers cleanup Lambda when a doc is deleted from sena/misty/orgs/"


def get_lambda_arn() -> str:
    response = lambda_.get_function(FunctionName=LAMBDA_NAME)
    arn = response["Configuration"]["FunctionArn"]
    print(f"Lambda ARN: {arn}")
    return arn


def create_eventbridge_rule() -> str:
    """
    Creates EventBridge rule that matches S3 ObjectRemoved events
    on sena-policy-docs bucket under sena/misty/orgs/ prefix.
    Returns the rule ARN.
    """
    event_pattern = {
        "source": ["aws.s3"],
        "detail-type": ["Object Deleted"],       # ObjectRemoved in EventBridge terms
        "detail": {
            "bucket": {
                "name": [BUCKET_NAME]
            },
            "object": {
                "key": [{"prefix": ORG_PREFIX}]
            }
        }
    }

    response = events.put_rule(
        Name=RULE_NAME,
        EventPattern=json.dumps(event_pattern),
        State="ENABLED",
        Description=RULE_DESCRIPTION
    )

    rule_arn = response["RuleArn"]
    print(f"EventBridge rule created: {rule_arn}")
    return rule_arn


def add_lambda_target(lambda_arn: str):
    """
    Adds the cleanup Lambda as the target of the EventBridge rule.
    """
    events.put_targets(
        Rule=RULE_NAME,
        Targets=[
            {
                "Id":  "cleanup-lambda",
                "Arn": lambda_arn
            }
        ]
    )
    print(f"Lambda target added to rule: {RULE_NAME}")


def grant_eventbridge_permission(lambda_arn: str, rule_arn: str):
    """
    Grants EventBridge permission to invoke the cleanup Lambda.
    Without this, EventBridge silently fails to invoke it.
    """
    try:
        lambda_.add_permission(
            FunctionName=LAMBDA_NAME,
            StatementId="eventbridge-s3-delete-invoke",
            Action="lambda:InvokeFunction",
            Principal="events.amazonaws.com",
            SourceArn=rule_arn
        )
        print("Lambda permission granted to EventBridge")
    except lambda_.exceptions.ResourceConflictException:
        print("Lambda permission already exists — skipping")


def verify_setup():
    """
    Reads back the rule and targets to confirm everything is wired correctly.
    """
    print("\n── Verification ─────────────────────────────────────────────")

    rule = events.describe_rule(Name=RULE_NAME)
    print(f"Rule name:    {rule['Name']}")
    print(f"Rule state:   {rule['State']}")
    print(f"Rule pattern: {rule['EventPattern']}")

    targets = events.list_targets_by_rule(Rule=RULE_NAME)
    for t in targets["Targets"]:
        print(f"Target ID:    {t['Id']}")
        print(f"Target ARN:   {t['Arn']}")

    print("─────────────────────────────────────────────────────────────")


def main():
    print("Setting up EventBridge rule for S3 delete → cleanup Lambda\n")

    # Step 1 — get Lambda ARN
    lambda_arn = get_lambda_arn()

    # Step 2 — create EventBridge rule
    rule_arn = create_eventbridge_rule()

    # Step 3 — add Lambda as target
    add_lambda_target(lambda_arn)

    # Step 4 — grant EventBridge permission to invoke Lambda
    grant_eventbridge_permission(lambda_arn, rule_arn)

    # Step 5 — verify
    verify_setup()

    print("\n✅ EventBridge delete rule setup complete")
    print("\nDelete flow is now fully wired:")
    print(f"  S3 delete from {ORG_PREFIX}*")
    print(f"  → EventBridge rule: {RULE_NAME}")
    print(f"  → Lambda: {LAMBDA_NAME}")
    print(f"  → metadata sidecar deleted")
    print(f"  → DELETE_NOT_FOUND ingestion job triggered")
    print(f"  → vectors and chunks removed from S3 Vectors")
    print(f"  → registry updated to DELETED")


if __name__ == "__main__":
    main()